# EDA

In [ ]:
# --- Imports
from pathlib import Path
import sys
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#---------- Reproducibility Configuration
sys.path.insert(0, str(Path("src").resolve()))
from reproducibility import DEFAULT_SEED, configure_reproducibility

SEED = DEFAULT_SEED
configure_reproducibility(SEED, include_tensorflow=False)
print(f"Global reproducibility seed configured: {SEED}")

# --- Config
DATA_DIR = Path("Data/1-Anonymization")
OUT_DIR  = Path("Outputs")
PLOTS_DIR = OUT_DIR / "eda_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = OUT_DIR / "summary.csv"

CONSOLIDATED_DIR = Path("Data/2-Consolidated")
CONSOLIDATED_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH  = CONSOLIDATED_DIR / "emg_consolidated.csv"
XLSX_PATH = CONSOLIDATED_DIR / "emg_consolidated.xlsx"
TXT_PATH  = CONSOLIDATED_DIR / "emg_consolidated.txt"

# --- Helpers
def parse_user_gesture(file_path: Path):
    """
    Extract user id and gesture from filename 'User-0007-g1.emt' (case-insensitive).
    Returns ('0007', 'g1').
    """
    m = re.search(r"User-(\d+)-g(\d+)\.emt$", file_path.name, flags=re.IGNORECASE)
    if not m:
        return ("unknown", "g?")
    return (m.group(1).zfill(4), f"g{m.group(2)}")

def _strip_df_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace in all string-like cells without FutureWarning."""
    try:
        return df.map(lambda x: x.strip() if isinstance(x, str) else x)
    except AttributeError:
        return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

def read_emt(file_path: Path):
    """
    Robust EMT reader:
    - Parse meta (Tracks, Frequency, Frames, Start time)
    - Detect header row and split column names with regex (tabs OR >=2 spaces)
    - Read data with explicit column names (header=None)
    - Keep rows with partial NaNs; drop only fully-empty rows/cols
    """
    with file_path.open("r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    meta = {}
    for line in lines[:200]:
        if "Tracks:" in line:
            m = re.search(r"Tracks:\s*(\d+)", line); 
            if m: meta["tracks"] = int(m.group(1))
        if "Frequency:" in line:
            m = re.search(r"Frequency:\s*([0-9]+)", line); 
            if m: meta["frequency_hz"] = int(m.group(1))
        if "Frames:" in line:
            m = re.search(r"Frames:\s*([0-9]+)", line); 
            if m: meta["frames"] = int(m.group(1))
        if "Start time:" in line:
            m = re.search(r"Start time:\s*([0-9.]+)", line); 
            if m: meta["start_time_s"] = float(m.group(1))

    # Find header line (starts with ' Frame' and contains 'Time')
    header_idx = None
    for i, line in enumerate(lines):
        if re.match(r"\s*Frame\s", line) and "Time" in line:
            header_idx = i
            break
    if header_idx is None:
        raise ValueError(f"Header line not found in {file_path}")

    # Parse column names from the header line explicitly
    sep_pattern = r"\t+|\s{2,}"
    header_line = lines[header_idx].strip()
    col_names = [c.strip() for c in re.split(sep_pattern, header_line) if c.strip()]

    # Read data from the next line using those exact column names
    df = pd.read_csv(
        file_path,
        sep=sep_pattern,
        engine="python",
        header=None,              # do not infer header
        names=col_names,          # use parsed names
        skiprows=header_idx + 1,  # start on first data row
        dtype=str                 # read as string first; will coerce later
    )

    # Cleanup: trim strings, convert blanks to NaN, drop fully empty rows/cols
    df = _strip_df_strings(df)
    df = df.replace(r"^\s*$", np.nan, regex=True)
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    # Coerce all columns to numeric (keep NaNs where conversion fails)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop leftover non-data rows (where BOTH Frame and Time are NaN)
    if "Frame" in df.columns and "Time" in df.columns:
        df = df[~(df["Frame"].isna() & df["Time"].isna())]

    df = df.reset_index(drop=True)

    # Safe duration computation
    if "Time" in df.columns:
        tt = df["Time"].dropna()
        if len(tt) >= 2:
            meta["duration_from_time_s"] = float(tt.iloc[-1] - tt.iloc[0])
    if "frames" in meta and "frequency_hz" in meta and meta.get("frames") and meta.get("frequency_hz"):
        meta["duration_from_meta_s"] = meta["frames"] / meta["frequency_hz"]

    return df, meta

def nice_title(text: str):
    """Short pretty title helper."""
    return text.replace("_", " ").strip()

def decimate_for_plot(df: pd.DataFrame, max_points: int = 50000) -> pd.DataFrame:
    """Uniform downsample for faster plotting."""
    if len(df) <= max_points:
        return df
    step = int(np.ceil(len(df) / max_points))
    return df.iloc[::step, :].reset_index(drop=True)

def safe_slug(s: str) -> str:
    """File-name friendly slug."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")

def save_figure_multi(basepath: Path, dpi_png: int = 160):
    """Save current figure as PNG and PDF (vector)."""
    plt.tight_layout()
    plt.savefig(basepath.with_name(basepath.name + ".png"), dpi=dpi_png)
    plt.savefig(basepath.with_name(basepath.name + ".pdf"))  # vector version
    plt.close()

def plot_single_signal(df, time_col, y_col, title, save_base: Path):
    """One chart per signal; saves PNG and PDF."""
    dfp = decimate_for_plot(df)
    plt.figure(figsize=(14, 4))
    plt.plot(dfp[time_col], dfp[y_col], linewidth=1.0)
    plt.grid(True, alpha=0.3)
    plt.xlabel("Time [s]")
    plt.ylabel("Amplitude [mV]")
    plt.title(nice_title(title))
    save_figure_multi(save_base)

def plot_stacked_signals(df, time_col, y_cols, title, save_base: Path):
    """All signals together (stacked); saves PNG and PDF."""
    dfp = decimate_for_plot(df)
    # Global offset based on robust amplitude
    amps = []
    for c in y_cols:
        s = dfp[c].values
        q = np.nanpercentile(np.abs(s), 95)
        amps.append(q if q > 0 else np.nanstd(s))
    step = np.nanmedian(amps) * 4.0 if len(amps) else 1.0

    plt.figure(figsize=(16, 9))
    for i, c in enumerate(y_cols):
        offset = i * step
        plt.plot(dfp[time_col], dfp[c] + offset, linewidth=0.8, label=c)
    plt.grid(True, alpha=0.3)
    plt.xlabel("Time [s]")
    plt.ylabel("Amplitude + offset [mV]")
    plt.title(nice_title(title))
    plt.legend(loc="upper right", fontsize=8, ncol=1, framealpha=0.9)
    save_figure_multi(save_base)

# --- Main load
emt_files = sorted(DATA_DIR.rglob("*.emt"))
if not emt_files:
    print(f"No .emt files found under {DATA_DIR.resolve()}. Please check the path.")

records = []            # summary rows
data_index = {}         # (user, gesture) -> info
consolidated_list = []  # for the big dataset

for fp in emt_files:
    user, gesture = parse_user_gesture(fp)
    df, meta = read_emt(fp)

    # Identify columns: first two are 'Frame', 'Time'; rest are signals
    cols = list(df.columns)
    if len(cols) < 3:
        print(f"Skipping {fp.name}: not enough columns.")
        continue
    time_col = "Time" if "Time" in cols else cols[1]
    signal_cols = [c for c in cols if c not in ("Frame", "Time")]

    # Store for later plotting
    key = (user, gesture)
    data_index[key] = {
        "df": df, "time_col": time_col, "signal_cols": signal_cols,
        "file": fp, "meta": meta
    }

    # Build summary row 
    rec = {
        "user": user,
        "gesture": gesture,
        "file": fp.name,
        "tracks_meta": meta.get("tracks"),
        "frequency_hz": meta.get("frequency_hz"),
        "frames_meta": meta.get("frames"),
        "duration_meta_s": meta.get("duration_from_meta_s"),
        "duration_time_s": meta.get("duration_from_time_s"),
        "n_rows_loaded": len(df)
    }
    records.append(rec)

    # Append to consolidated dataset
    df_con = df.copy()
    df_con.insert(0, "user", user)
    df_con.insert(1, "gesture", gesture)
    consolidated_list.append(df_con)

# --- Summary table (internal)
summary_df = pd.DataFrame.from_records(records)
summary_df = summary_df.sort_values(["user", "gesture"]).reset_index(drop=True)

# Compare durations if both available (internal check)
if not summary_df.empty:
    summary_df["duration_diff_s"] = (
        (summary_df["duration_meta_s"] - summary_df["duration_time_s"]).abs()
    )
    summary_df["duration_diff_ms"] = (summary_df["duration_diff_s"] * 1000).round(2)

# --- Public report
cols_to_drop = ["duration_meta_s", "n_rows_loaded", "duration_best_s"]
summary_out = summary_df.drop(columns=cols_to_drop, errors="ignore")

# Save summary CSV
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary_out.to_csv(SUMMARY_CSV, index=False)
print(f"Summary saved -> {SUMMARY_CSV.resolve()}")

# --- Consolidate all data (CSV, XLSX, TXT)
if consolidated_list:
    big_df = pd.concat(consolidated_list, ignore_index=True, sort=False)

    # Preferred column order: user, gesture, Frame, Time, then the rest
    base_cols = ["user", "gesture"]
    ordered_cols = base_cols + [c for c in ["Frame", "Time"] if c in big_df.columns]
    other_cols = [c for c in big_df.columns if c not in ordered_cols]
    big_df = big_df[ordered_cols + other_cols]

    # Save CSV
    big_df.to_csv(CSV_PATH, index=False)

    # Save TXT (tab-delimited)
    big_df.to_csv(TXT_PATH, index=False, sep="\t")

    # Save XLSX (auto-split if > 1,000,000 rows)
    max_rows_excel = 1_000_000
    with pd.ExcelWriter(XLSX_PATH) as writer:
        if len(big_df) <= max_rows_excel:
            big_df.to_excel(writer, index=False, sheet_name="All")
        else:
            start = 0
            part = 1
            while start < len(big_df):
                end = min(start + max_rows_excel, len(big_df))
                chunk = big_df.iloc[start:end]
                chunk.to_excel(writer, index=False, sheet_name=f"Part_{part}")
                start = end
                part += 1

    print(f"Consolidated CSV -> {CSV_PATH.resolve()}")
    print(f"Consolidated XLSX -> {XLSX_PATH.resolve()}")
    print(f"Consolidated TXT -> {TXT_PATH.resolve()}")
else:
    print("No data found to consolidate.")

# --- Plotting
for (user, gesture), pack in data_index.items():
    df = pack["df"]; time_col = pack["time_col"]; signal_cols = pack["signal_cols"]

    # Output folders
    base = PLOTS_DIR / f"user_{user}" / f"{gesture}"
    base.mkdir(parents=True, exist_ok=True)

    # 1) Each signal individually -> saves .png + .pdf
    for sig in signal_cols:
        title = f"User {user} | {gesture.upper()} | Signal: {sig}"
        save_base = base / f"{gesture}_{safe_slug(sig)}"
        plot_single_signal(df, time_col, sig, title, save_base)

    # 2) All signals together (stacked) -> saves .png + .pdf
    title = f"User {user} | {gesture.upper()} | All signals (stacked)"
    save_base = base / f"{gesture}_ALL_SIGNALS"
    plot_stacked_signals(df, time_col, signal_cols, title, save_base)

print(f"Plots saved under -> {PLOTS_DIR.resolve()}")
display(summary_out.head(20))
